手写MHA

In [ ]:
"""
Shape:
B = batch size
T = sequence length
C = hidden size
H = num_heads
D = head_dim = C / H

完整流程：
x
-> 线性变换得到Q, K, V
-> reshape成多头形式
-> 计算attention score: QK^T / sqrt(d_head)
-> 加causal mask
-> softmax
-> 乘V
-> 多头concat
-> output projection

"""

'\nShape:\nB = batch size\nT = sequence length\nC = hidden size\nH = num_heads\nD = head_dim = C / H\n'

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        assert hidden_size % num_heads == 0
        
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.o_proj = nn.Linear(hidden_size, hidden_size)
    
    def forward(self, x, causal=True):
        B, T, C = x.shape
        
        # 1. 线性映射得到 q, k, v
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        
        # 2. reshape 成多头形式: [B, T, C] -> [B, H, T, D]
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        
        # 3. attention scores: [B, H, T, T]
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim) 
        
        # 4. causal mask
        if causal:
            mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
            scores = scores.masked_fill(mask, float("-inf"))
        
        # 5. softmax
        attn = F.softmax(scores, dim=-1)
        
        # 6. 加权求和 V
        out = attn @ v
        
        # 7. 合并 head: [B, H, T, D] -> [B, T, C]
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        
        # 8. 输出投影
        out = self.o_proj(out)
        
        return out

In [7]:
x = torch.randn(2, 4, 16)
mha = MultiHeadAttention(hidden_size=16, num_heads=4)
y = mha(x)

print(y.shape)

torch.Size([2, 4, 16])
